In [1]:
import torch
import kornia.feature as KF
import os
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

HPATCHES_PATH = "../hpatches-sequences-release"


c:\Users\EasyMoneySniper\anaconda3\envs\nlp-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Check the layers of the LoFTR model
teacher = KF.LoFTR(pretrained='outdoor').eval()
for name, module in teacher.named_modules():
    print(name)


backbone
backbone.conv1
backbone.bn1
backbone.relu
backbone.layer1
backbone.layer1.0
backbone.layer1.0.conv1
backbone.layer1.0.conv2
backbone.layer1.0.bn1
backbone.layer1.0.bn2
backbone.layer1.0.relu
backbone.layer1.1
backbone.layer1.1.conv1
backbone.layer1.1.conv2
backbone.layer1.1.bn1
backbone.layer1.1.bn2
backbone.layer1.1.relu
backbone.layer2
backbone.layer2.0
backbone.layer2.0.conv1
backbone.layer2.0.conv2
backbone.layer2.0.bn1
backbone.layer2.0.bn2
backbone.layer2.0.relu
backbone.layer2.0.downsample
backbone.layer2.0.downsample.0
backbone.layer2.0.downsample.1
backbone.layer2.1
backbone.layer2.1.conv1
backbone.layer2.1.conv2
backbone.layer2.1.bn1
backbone.layer2.1.bn2
backbone.layer2.1.relu
backbone.layer3
backbone.layer3.0
backbone.layer3.0.conv1
backbone.layer3.0.conv2
backbone.layer3.0.bn1
backbone.layer3.0.bn2
backbone.layer3.0.relu
backbone.layer3.0.downsample
backbone.layer3.0.downsample.0
backbone.layer3.0.downsample.1
backbone.layer3.1
backbone.layer3.1.conv1
backbone.la

In [3]:
# loftr_coarse
# coarse_matching

In [3]:
# Check the outputs of the coarse features and confidence matrix
teacher = KF.LoFTR(pretrained='outdoor').eval().cuda()

# use hooks to capture the outputs of the layers we care about
teacher_cache = {}
teacher.loftr_coarse.register_forward_hook(
    lambda m, inp, out: teacher_cache.update({'coarse_feat': out})
)
teacher.coarse_matching.register_forward_hook(
    lambda m, inp, out: teacher_cache.update({'conf_matrix': out})
)

# create dummy input images
img0 = torch.rand(1, 1, 480, 480).cuda()
img1 = torch.rand(1, 1, 480, 480).cuda()

with torch.no_grad():
    t_out = teacher({"image0": img0, "image1": img1})

# check what we captured in the cache
for key, val in teacher_cache.items():
    if isinstance(val, torch.Tensor):
        print(f"{key}: shape={val.shape}, dtype={val.dtype}")
    elif isinstance(val, (tuple, list)):
        for i, v in enumerate(val):
            if isinstance(v, torch.Tensor):
                print(f"{key}[{i}]: shape={v.shape}")
            else:
                print(f"{key}[{i}]: type={type(v)}")
    elif isinstance(val, dict):
        for k, v in val.items():
            if isinstance(v, torch.Tensor):
                print(f"{key}['{k}']: shape={v.shape}")
    else:
        print(f"{key}: type={type(val)}")

coarse_feat[0]: shape=torch.Size([1, 3600, 256])
coarse_feat[1]: shape=torch.Size([1, 3600, 256])
conf_matrix: type=<class 'NoneType'>


From the above result, we noticed that coarse_feat is valid, since it's a tuple with two tensors inside, which is just the coarse feature after img0 and img1 going through the transformer, shape is (B, 3600, 256) (matches: B, H*W, C), 3600 = 60 * 60, 256 is the shape of LoFTR coarse. 

However, conf_matrix has a shape of None. Since Kornia's coarse_matching does not return conf_matrix directly. We need to calculate conf_matrix from coarse_feat without hook. LofTR's dual-softmax was conducted based on the two featrues and then do softmax. 

In [4]:
# We can see that the coarse features have shape (B, 3600, 256) and the confidence matrix has shape (B, 3600, 3600).
# The confidence matrix is computed from the coarse features, so we can verify that the teacher's confidence matrix is consistent with the coarse features.
# We can calculate the teacher confidence matrix from the coarse features and compare it with the one captured from the coarse_matching layer.
import torch.nn.functional as F
'''
This is the code to calculate the teacher confidence matrix from the coarse features. 
You can run this code to verify that the teacher's confidence matrix is consistent with the coarse features. 
The teacher confidence matrix should be similar to the one captured from the coarse_matching layer, which means our understanding of how the teacher computes the confidence matrix is correct.
'''
teacher.loftr_coarse.register_forward_hook(
    lambda m, inp, out: teacher_cache.update({
        'coarse_feat0': out[0],  # (B, 3600, 256)
        'coarse_feat1': out[1],  # (B, 3600, 256)
    })
)

# forward
with torch.no_grad():
    t_out = teacher({"image0": img0, "image1": img1})

# calculate the teacher confidence matrix
feat0 = F.normalize(teacher_cache['coarse_feat0'], dim=-1)
feat1 = F.normalize(teacher_cache['coarse_feat1'], dim=-1)
teacher_conf = torch.bmm(feat0, feat1.transpose(1, 2)) / 0.1  # temperature=0.1
teacher_conf = F.softmax(teacher_conf, dim=-1) * F.softmax(teacher_conf, dim=-2)
# teacher_conf: (B, 3600, 3600)

In [6]:
# Demo note only (no execution needed)
# Student feature KD alignment idea:
#   student coarse_feat0: (B, 128, 60, 60)
#   flatten + permute -> (B, 3600, 128)
#   linear projection  -> (B, 3600, 256)
#   compare with teacher coarse_feat0 using MSE
#
# Real implementation is already included in the training loop cells below.

In [5]:
# Training loop (cell 1/3): imports + setup + helper builders
import sys
import csv
import argparse
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torch.amp import autocast, GradScaler

# Resolve project root robustly for both cwd=main and cwd=main/notebooks
_cwd = Path.cwd().resolve()
_candidates = [_cwd, _cwd.parent]
PROJECT_ROOT = None
for _p in _candidates:
    if (_p / "dataset.py").exists() and (_p / "models").exists() and (_p / "utils" / "geometry.py").exists():
        PROJECT_ROOT = _p
        break
if PROJECT_ROOT is None:
    raise RuntimeError("Cannot locate project root that contains dataset.py/models/utils/geometry.py")

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "utils"))

from dataset import SyntheticHomographyDataset
from models import StudentHybrid, StudentCNN, StudentHybridConfig, StudentCNNConfig
from losses import DistillationLoss
from geometry import create_ground_truth_from_homography  # pyright: ignore[reportMissingImports]


def parse_args_notebook():
    parser = argparse.ArgumentParser()
    parser.add_argument("--student_type", type=str, default="all", choices=["all", "hybrid", "cnn"])
    parser.add_argument("--hpatches_path", type=str, default=HPATCHES_PATH)
    parser.add_argument("--num_pairs", type=int, default=10000)
    parser.add_argument("--val_ratio", type=float, default=0.1)
    parser.add_argument("--batch_size", type=int, default=4)
    parser.add_argument("--epochs", type=int, default=30)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--weight_decay", type=float, default=1e-2)
    parser.add_argument("--patch_size", type=int, default=480)
    parser.add_argument("--num_workers", type=int, default=0)
    parser.add_argument("--amp", action="store_true", default=True)
    parser.add_argument("--temperature", type=float, default=0.1)
    parser.add_argument("--output_dir", type=str, default="../checkpoints")
    args, _ = parser.parse_known_args()
    return args


def build_teacher(device):
    teacher = KF.LoFTR(pretrained="outdoor").eval().to(device)
    for p in teacher.parameters():
        p.requires_grad = False

    teacher_cache = {}

    def _coarse_hook(_module, _inp, out):
        teacher_cache["coarse_feat0"] = out[0].detach()  # (B, 3600, 256)
        teacher_cache["coarse_feat1"] = out[1].detach()  # (B, 3600, 256)

    teacher.loftr_coarse.register_forward_hook(_coarse_hook)
    return teacher, teacher_cache


@torch.no_grad()
def get_teacher_targets(teacher, teacher_cache, img0, img1, temperature=0.1):
    _ = teacher({"image0": img0, "image1": img1})

    feat0 = F.normalize(teacher_cache["coarse_feat0"], dim=-1)
    feat1 = F.normalize(teacher_cache["coarse_feat1"], dim=-1)

    sim = torch.bmm(feat0, feat1.transpose(1, 2)) / temperature
    teacher_conf = F.softmax(sim, dim=-1) * F.softmax(sim, dim=-2)

    return {
        "conf_matrix": teacher_conf,
        "teacher_coarse_feat0": teacher_cache["coarse_feat0"],
        "teacher_coarse_feat1": teacher_cache["coarse_feat1"],
    }


def build_dataloaders(args):
    dataset = SyntheticHomographyDataset(
        image_dir=args.hpatches_path,
        num_pairs=args.num_pairs,
        patch_size=args.patch_size,
    )

    val_len = int(len(dataset) * args.val_ratio)
    train_len = len(dataset) - val_len
    split_gen = torch.Generator().manual_seed(42)
    train_set, val_set = random_split(dataset, [train_len, val_len], generator=split_gen)

    train_loader = DataLoader(
        train_set,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_set,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    return train_loader, val_loader


def make_student(student_kind, device, only_gt=False):
    if student_kind == "hybrid":
        config = StudentHybridConfig()
        model = StudentHybrid(config).to(device)
    elif student_kind == "cnn":
        config = StudentCNNConfig()
        model = StudentCNN(config).to(device)
    else:
        raise ValueError(f"Unsupported student type: {student_kind}")

    # Fine-level KD is intentionally disabled in this project setup.
    config.lambda_fine_kd = 0.0

    if only_gt:
        config.lambda_coarse_kd = 0.0
        config.lambda_feat_kd = 0.0
        config.lambda_gt = 1.0

    feature_projector = nn.Linear(config.coarse_dim, config.teacher_coarse_dim).to(device)
    return model, config, feature_projector


def project_student_feat_for_kd(feat_map, projector):
    feat_seq = feat_map.flatten(2).permute(0, 2, 1)  # (B, 3600, 128)
    return projector(feat_seq)  # (B, 3600, 256)

In [6]:
# Training loop (cell 2/3): epoch routine + per-run trainer
import time

def train_or_val_one_epoch(
    model,
    criterion,
    optimizer,
    scaler,
    teacher,
    teacher_cache,
    feature_projector,
    loader,
    device,
    temperature,
    train_mode=True,
):
    meter = {"total": 0.0, "coarse_kd": 0.0, "feat_kd": 0.0, "gt_sup": 0.0}

    if train_mode:
        model.train()
    else:
        model.eval()

    phase = "train" if train_mode else "val"
    pbar = tqdm(loader, desc=f"{phase}", leave=False)

    context = torch.enable_grad() if train_mode else torch.no_grad()
    with context:
        for img0, img1, H_gt in pbar:
            img0 = img0.to(device, non_blocking=True)
            img1 = img1.to(device, non_blocking=True)
            H_gt = H_gt.to(device, non_blocking=True)

            teacher_targets = get_teacher_targets(teacher, teacher_cache, img0, img1, temperature=temperature)
            gt_conf = create_ground_truth_from_homography(
                H_gt, height=img0.shape[-2], width=img0.shape[-1], coarse_scale=8
            )

            if train_mode:
                optimizer.zero_grad(set_to_none=True)

            with autocast(device_type=device.type, enabled=(scaler.is_enabled() and train_mode)):
                student_data = model({"image0": img0, "image1": img1})

                student_data["coarse_feat0_proj"] = project_student_feat_for_kd(
                    student_data["coarse_feat0"], feature_projector
                )
                student_data["coarse_feat1_proj"] = project_student_feat_for_kd(
                    student_data["coarse_feat1"], feature_projector
                )

                loss_dict = criterion(
                    student_data,
                    {
                        "conf_matrix": teacher_targets["conf_matrix"],
                        "teacher_coarse_feat0": teacher_targets["teacher_coarse_feat0"],
                        "teacher_coarse_feat1": teacher_targets["teacher_coarse_feat1"],
                    },
                    {"gt_conf_matrix": gt_conf},
                )

            if train_mode:
                scaler.scale(loss_dict["total"]).backward()
                scaler.step(optimizer)
                scaler.update()

            for k in meter.keys():
                meter[k] += float(loss_dict[k].detach().item())

            done = max(1, pbar.n)
            pbar.set_postfix({
                "total": f"{meter['total'] / done:.4f}",
                "feat": f"{meter['feat_kd'] / done:.4f}",
            })

    num_batches = max(1, len(loader))
    for k in meter:
        meter[k] /= num_batches
    return meter


def run_one_training(
    run_name,
    student_kind,
    only_gt,
    args,
    device,
    teacher,
    teacher_cache,
    train_loader,
    val_loader,
):
    model, config, feature_projector = make_student(student_kind, device, only_gt=only_gt)
    criterion = DistillationLoss(config).to(device)

    optimizer = torch.optim.AdamW(
        list(model.parameters()) + list(feature_projector.parameters()),
        lr=args.lr,
        weight_decay=args.weight_decay,
    )

    scaler = GradScaler("cuda", enabled=(args.amp and device.type == "cuda"))

    run_dir = Path(args.output_dir) / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    log_path = run_dir / "epoch_losses.csv"

    with open(log_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch",
            "train_total", "train_coarse_kd", "train_feat_kd", "train_gt_sup",
            "val_total", "val_coarse_kd", "val_feat_kd", "val_gt_sup",
        ])

    run_start = time.time()
    for epoch in range(1, args.epochs + 1):
        epoch_start = time.time()

        train_meter = train_or_val_one_epoch(
            model=model,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            teacher=teacher,
            teacher_cache=teacher_cache,
            feature_projector=feature_projector,
            loader=train_loader,
            device=device,
            temperature=args.temperature,
            train_mode=True,
        )

        val_meter = train_or_val_one_epoch(
            model=model,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            teacher=teacher,
            teacher_cache=teacher_cache,
            feature_projector=feature_projector,
            loader=val_loader,
            device=device,
            temperature=args.temperature,
            train_mode=False,
        )

        epoch_time = time.time() - epoch_start
        elapsed = time.time() - run_start
        progress = (epoch / args.epochs) * 100.0
        avg_epoch = elapsed / epoch
        eta_sec = avg_epoch * (args.epochs - epoch)
        eta_min = eta_sec / 60.0

        print(
            f"[{run_name}] Epoch {epoch:02d}/{args.epochs} ({progress:.1f}%) | "
            f"epoch={epoch_time/60.0:.2f} min, ETA={eta_min:.1f} min | "
            f"train total={train_meter['total']:.4f}, coarse={train_meter['coarse_kd']:.4f}, "
            f"feat={train_meter['feat_kd']:.4f}, gt={train_meter['gt_sup']:.4f} | "
            f"val total={val_meter['total']:.4f}, coarse={val_meter['coarse_kd']:.4f}, "
            f"feat={val_meter['feat_kd']:.4f}, gt={val_meter['gt_sup']:.4f}"
        )

        with open(log_path, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                epoch,
                train_meter["total"], train_meter["coarse_kd"], train_meter["feat_kd"], train_meter["gt_sup"],
                val_meter["total"], val_meter["coarse_kd"], val_meter["feat_kd"], val_meter["gt_sup"],
            ])

        ckpt_path = run_dir / f"epoch_{epoch:02d}.pth"
        torch.save(
            {
                "epoch": epoch,
                "run_name": run_name,
                "student_kind": student_kind,
                "model_state_dict": model.state_dict(),
                "projector_state_dict": feature_projector.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "config": config.__dict__,
                "train_loss": train_meter,
                "val_loss": val_meter,
            },
            ckpt_path,
        )

    return model

In [9]:
# Training loop (cell 3/4): smoke test only
args = parse_args_notebook()
args.num_pairs = 200
args.epochs = 1
args.student_type = "hybrid"

print(
    f"Smoke test config | student_type={args.student_type}, "
    f"num_pairs={args.num_pairs}, epochs={args.epochs}, batch_size={args.batch_size}"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_loader, val_loader = build_dataloaders(args)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

teacher, teacher_cache = build_teacher(device)

run_plan = [("run_smoke_hybrid", "hybrid", False)]
for run_name, student_kind, only_gt in run_plan:
    _ = run_one_training(
        run_name=run_name,
        student_kind=student_kind,
        only_gt=only_gt,
        args=args,
        device=device,
        teacher=teacher,
        teacher_cache=teacher_cache,
        train_loader=train_loader,
        val_loader=val_loader,
    )

print("Smoke test finished.")

Smoke test config | student_type=hybrid, num_pairs=200, epochs=1, batch_size=4
Device: cuda
Train batches: 45, Val batches: 5


[run_smoke_hybrid] Epoch 01/1 (100.0%) | epoch=24.42 min, ETA=0.0 min | train total=8.5232, coarse=3.4506, feat=4.9539, gt=2.5956 | val total=10.0669, coarse=4.3397, feat=4.8648, gt=3.2948
Smoke test finished.


In [7]:
# Pre-training sanity checks — run this cell before ANY training cell.
# Covers all 8 checklist items: teacher freeze, hook shapes, student alignment,
# GT shape, loss sanity, gradient flow, run-config parity, and checkpoint round-trip.

print("=" * 60)
print("PRE-TRAINING SANITY CHECKS")
print("=" * 60)

_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── 1. Teacher is frozen ──────────────────────────────────────────────────────
_teacher_chk, _teacher_cache_chk = build_teacher(_device)
assert not _teacher_chk.training, "FAIL: teacher is not in eval() mode"
assert not any(p.requires_grad for p in _teacher_chk.parameters()), \
    "FAIL: teacher has trainable parameters"
print("[1] Teacher frozen: eval() ✓   requires_grad=False on all params ✓")

# ── 2. Hook output shapes ─────────────────────────────────────────────────────
_args_chk = parse_args_notebook()
_args_chk.num_pairs = 20
_args_chk.batch_size = 2
_chk_loader, _ = build_dataloaders(_args_chk)
_img0_chk, _img1_chk, _H_chk = next(iter(_chk_loader))
_img0_chk = _img0_chk.to(_device)
_img1_chk = _img1_chk.to(_device)
_H_chk    = _H_chk.to(_device)

_t_targets = get_teacher_targets(_teacher_chk, _teacher_cache_chk, _img0_chk, _img1_chk)

_cf0_shape = _teacher_cache_chk["coarse_feat0"].shape
print(f"[2] teacher_cache['coarse_feat0'].shape = {_cf0_shape}")
assert list(_cf0_shape) == [_args_chk.batch_size, 3600, 256], \
    f"FAIL: expected (B,3600,256), got {_cf0_shape}"

_tc_shape = _t_targets["conf_matrix"].shape
print(f"[2] teacher_conf.shape               = {_tc_shape}")
assert list(_tc_shape) == [_args_chk.batch_size, 3600, 3600], \
    f"FAIL: expected (B,3600,3600), got {_tc_shape}"

# ── 3. Student-teacher feature alignment ─────────────────────────────────────
_model_chk, _config_chk, _proj_chk = make_student("hybrid", _device, only_gt=False)
_model_chk.train()
with torch.enable_grad():
    _s_out = _model_chk({"image0": _img0_chk, "image1": _img1_chk})

_feat_map = _s_out["coarse_feat0"]
print(f"[3] coarse_feat0 before flatten+permute : {_feat_map.shape}")
assert list(_feat_map.shape) == [_args_chk.batch_size, 128, 60, 60], \
    f"FAIL: expected (B,128,60,60), got {_feat_map.shape}"

_feat_seq = _feat_map.flatten(2).permute(0, 2, 1)
print(f"[3] coarse_feat0 after  flatten+permute : {_feat_seq.shape}")
assert list(_feat_seq.shape) == [_args_chk.batch_size, 3600, 128], \
    f"FAIL: expected (B,3600,128), got {_feat_seq.shape}"

_feat_proj_chk = _proj_chk(_feat_seq)
print(f"[3] coarse_feat0 after  projector       : {_feat_proj_chk.shape}")
assert list(_feat_proj_chk.shape) == [_args_chk.batch_size, 3600, 256], \
    f"FAIL: expected (B,3600,256), got {_feat_proj_chk.shape}"

# ── 4. GT matrix shape matches student conf_matrix ───────────────────────────
_gt_matrix = create_ground_truth_from_homography(
    _H_chk, height=_img0_chk.shape[-2], width=_img0_chk.shape[-1], coarse_scale=8
)
print(f"[4] gt_matrix.shape   = {_gt_matrix.shape}")
print(f"[4] conf_matrix.shape = {_s_out['conf_matrix'].shape}")
assert _gt_matrix.shape == _s_out["conf_matrix"].shape, \
    f"FAIL: shape mismatch {_gt_matrix.shape} vs {_s_out['conf_matrix'].shape}"

# ── 5. Loss values reasonable on first batch ──────────────────────────────────
_crit_chk = DistillationLoss(_config_chk).to(_device)
_s_out["coarse_feat0_proj"] = project_student_feat_for_kd(_s_out["coarse_feat0"], _proj_chk)
_s_out["coarse_feat1_proj"] = project_student_feat_for_kd(_s_out["coarse_feat1"], _proj_chk)

_loss_chk = _crit_chk(
    _s_out,
    {"conf_matrix":           _t_targets["conf_matrix"],
     "teacher_coarse_feat0":  _t_targets["teacher_coarse_feat0"],
     "teacher_coarse_feat1":  _t_targets["teacher_coarse_feat1"]},
    {"gt_conf_matrix": _gt_matrix},
)
print("[5] Loss components (first batch):")
for _k, _v in _loss_chk.items():
    _val = float(_v)
    assert _val == _val,          f"FAIL: {_k} is NaN"
    assert _val != float("inf"),  f"FAIL: {_k} is inf"
    print(f"     {_k:12s}: {_val:.4f}")

# ── 6. Gradient flow ──────────────────────────────────────────────────────────
_loss_chk["total"].backward()

_bb_grad = _model_chk.backbone.coarse_proj.weight.grad
assert _bb_grad is not None, "FAIL: backbone.coarse_proj has no gradient"
print(f"[6] backbone.coarse_proj.weight.grad  norm = {_bb_grad.norm():.4f}  ✓")

_lp_grad = _proj_chk.weight.grad
assert _lp_grad is not None, "FAIL: feature_projector (Linear) has no gradient"
print(f"[6] feature_projector(Linear).weight.grad norm = {_lp_grad.norm():.4f}  ✓")

assert not any(p.grad is not None for p in _teacher_chk.parameters()), \
    "FAIL: teacher has gradients — must not appear in the backward graph"
print("[6] Teacher gradients: none ✓")

# ── 7. Run config parity (GT-only vs KD-run) ──────────────────────────────────
_, _cfg_kd, _ = make_student("hybrid", _device, only_gt=False)
_, _cfg_gt, _ = make_student("hybrid", _device, only_gt=True)

assert _cfg_gt.lambda_coarse_kd == 0.0, "FAIL: GT-only run has non-zero lambda_coarse_kd"
assert _cfg_gt.lambda_feat_kd   == 0.0, "FAIL: GT-only run has non-zero lambda_feat_kd"
assert _cfg_gt.lambda_gt        == 1.0, "FAIL: GT-only run has wrong lambda_gt"
for _field in ("coarse_dim", "teacher_coarse_dim", "temperature", "match_threshold"):
    assert getattr(_cfg_kd, _field) == getattr(_cfg_gt, _field), \
        f"FAIL: shared field '{_field}' differs between KD and GT-only configs"
print("[7] Run config parity: GT-only zeros KD weights, all shared fields identical ✓")

print("=" * 60)
print("ALL SANITY CHECKS PASSED — safe to proceed to training cells.")
print("Note: checkpoint round-trip check (item 8) runs after full training.")
print("=" * 60)


PRE-TRAINING SANITY CHECKS
[1] Teacher frozen: eval() ✓   requires_grad=False on all params ✓
[2] teacher_cache['coarse_feat0'].shape = torch.Size([2, 3600, 256])
[2] teacher_conf.shape               = torch.Size([2, 3600, 3600])
[3] coarse_feat0 before flatten+permute : torch.Size([2, 128, 60, 60])
[3] coarse_feat0 after  flatten+permute : torch.Size([2, 3600, 128])
[3] coarse_feat0 after  projector       : torch.Size([2, 3600, 256])
[4] gt_matrix.shape   = torch.Size([2, 3600, 3600])
[4] conf_matrix.shape = torch.Size([2, 3600, 3600])
[5] Loss components (first batch):
     coarse_kd   : 3.9849
     feat_kd     : 5.2811
     fine_kd     : 0.0000
     gt_sup      : 2.6931
     total       : 9.3185
[6] backbone.coarse_proj.weight.grad  norm = 1.0127  ✓
[6] feature_projector(Linear).weight.grad norm = 0.1892  ✓
[6] Teacher gradients: none ✓
[7] Run config parity: GT-only zeros KD weights, all shared fields identical ✓
ALL SANITY CHECKS PASSED — safe to proceed to training cells.
Note: c

In [11]:
# Training loop (cell 4/4): practical full training plan
args = parse_args_notebook()

# Keep runtime practical for a 5-day deadline.
args.num_pairs = 1000
args.epochs = 5
args.batch_size = 4
args.student_type = "hybrid"

print(
    f"Full run config | student_type={args.student_type}, "
    f"num_pairs={args.num_pairs}, epochs={args.epochs}, batch_size={args.batch_size}"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_loader, val_loader = build_dataloaders(args)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

teacher, teacher_cache = build_teacher(device)

# Three comparison runs — identical config except the intended difference:
#   Run 1 (run_full_hybrid)    : Hybrid student, full KD (coarse_kd + feat_kd + gt_sup)
#   Run 2 (run_full_cnn)       : CNN student,    full KD (coarse_kd + feat_kd + gt_sup)
#   Run 3 (run_full_hybrid_gt) : Hybrid student, GT supervision only (KD weights = 0)
run_plan = [
    ("run_full_hybrid",    "hybrid", False),   # Run 1: KD distillation
    ("run_full_cnn",       "cnn",    False),   # Run 2: CNN baseline with KD
    ("run_full_hybrid_gt", "hybrid", True),    # Run 3: GT-only (ablation)
]

for run_name, student_kind, only_gt in run_plan:
    print(f"\n{'='*60}")
    print(f"Starting {run_name}  (student={student_kind}, gt_only={only_gt})")
    print(f"{'='*60}")
    _ = run_one_training(
        run_name=run_name,
        student_kind=student_kind,
        only_gt=only_gt,
        args=args,
        device=device,
        teacher=teacher,
        teacher_cache=teacher_cache,
        train_loader=train_loader,
        val_loader=val_loader,
    )

print("\nFull training runs finished.")


Full run config | student_type=hybrid, num_pairs=1000, epochs=5, batch_size=4
Device: cuda
Train batches: 225, Val batches: 25

Starting run_full_hybrid  (student=hybrid, gt_only=False)


[run_full_hybrid] Epoch 01/5 (20.0%) | epoch=51.63 min, ETA=206.5 min | train total=7.1253, coarse=2.6461, feat=4.5147, gt=2.2218 | val total=6.8290, coarse=2.4443, feat=4.2400, gt=2.2647


[run_full_hybrid] Epoch 02/5 (40.0%) | epoch=66.45 min, ETA=177.1 min | train total=5.5439, coarse=1.8392, feat=3.8526, gt=1.7784 | val total=5.2134, coarse=1.6857, feat=3.6278, gt=1.7139


[run_full_hybrid] Epoch 03/5 (60.0%) | epoch=52.93 min, ETA=114.0 min | train total=4.8310, coarse=1.5555, feat=3.3983, gt=1.5764 | val total=4.5422, coarse=1.4633, feat=3.2032, gt=1.4773


[run_full_hybrid] Epoch 04/5 (80.0%) | epoch=52.08 min, ETA=55.8 min | train total=4.4071, coarse=1.4013, feat=3.1131, gt=1.4493 | val total=4.2396, coarse=1.2862, feat=3.0720, gt=1.4173


...

[run_full_hybrid_gt] Epoch 01/5 (20.0%) | epoch=52.57 min, ETA=210.3 min | train total=2.0871, coarse=3.1540, feat=5.1667, gt=2.0871 | val total=2.0171, coarse=2.8224, feat=5.0856, gt=2.0171


[run_full_hybrid_gt] Epoch 02/5 (40.0%) | epoch=52.80 min, ETA=158.0 min | train total=1.5483, coarse=2.5331, feat=5.1747, gt=1.5483 | val total=1.3868, coarse=2.4874, feat=5.1782, gt=1.3868


[run_full_hybrid_gt] Epoch 03/5 (60.0%) | epoch=53.54 min, ETA=105.9 min | train total=1.2524, coarse=2.3777, feat=5.2056, gt=1.2524 | val total=1.1052, coarse=2.2512, feat=5.2283, gt=1.1052


[run_full_hybrid_gt] Epoch 04/5 (80.0%) | epoch=51.11 min, ETA=52.5 min | train total=1.1031, coarse=2.2925, feat=5.2026, gt=1.1031 | val total=1.0914, coarse=2.3518, feat=5.2057, gt=1.0914


[run_full_hybrid_gt] Epoch 05/5 (100.0%) | epoch=79.50 min, ETA=0.0 min | train total=1.0270, coarse=2.2506, feat=5.2031, gt=1.0270 | val total=0.9544, coarse=2.2627, feat=5.2365, gt=0.9544

Full training runs finished.


In [ ]:
# Resume training from epoch 5 → 20 for all three runs
import gc

RESUME_FROM_EPOCH = 5
TARGET_EPOCHS     = 20

# ── Evict any models left on GPU from previous cells ─────────────────────────
for _var in ["teacher", "model", "feature_projector", "criterion", "optimizer", "scaler"]:
    if _var in dir():
        del globals()[_var]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1024**2:.0f} MB allocated, "
          f"{torch.cuda.memory_reserved()/1024**2:.0f} MB reserved")

args_r = parse_args_notebook()
args_r.num_pairs  = 1000
args_r.epochs     = TARGET_EPOCHS
args_r.batch_size = 4   # match original run — hybrid attention is memory-heavy
args_r.amp        = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_loader, val_loader = build_dataloaders(args_r)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

teacher, teacher_cache = build_teacher(device)

resume_plan = [
    ("run_full_hybrid",    "hybrid", False),
    ("run_full_cnn",       "cnn",    False),
    ("run_full_hybrid_gt", "hybrid", True),
]

for run_name, student_kind, only_gt in resume_plan:
    ckpt_path = Path(args_r.output_dir) / run_name / f"epoch_{RESUME_FROM_EPOCH:02d}.pth"
    assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    print(f"\n{'='*60}")
    print(f"Resuming {run_name} from epoch {RESUME_FROM_EPOCH} → {TARGET_EPOCHS}")
    print(f"{'='*60}")

    model, config, feature_projector = make_student(student_kind, device, only_gt=only_gt)
    for k, v in ckpt["config"].items():
        if hasattr(config, k):
            setattr(config, k, v)

    model.load_state_dict(ckpt["model_state_dict"])
    feature_projector.load_state_dict(ckpt["projector_state_dict"])

    criterion = DistillationLoss(config).to(device)
    optimizer = torch.optim.AdamW(
        list(model.parameters()) + list(feature_projector.parameters()),
        lr=args_r.lr,
        weight_decay=args_r.weight_decay,
    )
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])

    scaler = GradScaler("cuda", enabled=(args_r.amp and device.type == "cuda"))

    run_dir  = Path(args_r.output_dir) / run_name
    log_path = run_dir / "epoch_losses.csv"

    run_start = time.time()
    for epoch in range(RESUME_FROM_EPOCH + 1, TARGET_EPOCHS + 1):
        epoch_start = time.time()

        train_meter = train_or_val_one_epoch(
            model=model, criterion=criterion, optimizer=optimizer, scaler=scaler,
            teacher=teacher, teacher_cache=teacher_cache,
            feature_projector=feature_projector, loader=train_loader,
            device=device, temperature=args_r.temperature, train_mode=True,
        )
        val_meter = train_or_val_one_epoch(
            model=model, criterion=criterion, optimizer=optimizer, scaler=scaler,
            teacher=teacher, teacher_cache=teacher_cache,
            feature_projector=feature_projector, loader=val_loader,
            device=device, temperature=args_r.temperature, train_mode=False,
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        epoch_time  = time.time() - epoch_start
        elapsed     = time.time() - run_start
        epochs_done = epoch - RESUME_FROM_EPOCH
        epochs_left = TARGET_EPOCHS - epoch
        eta_min = (elapsed / epochs_done) * epochs_left / 60.0

        print(
            f"[{run_name}] Epoch {epoch:02d}/{TARGET_EPOCHS} | "
            f"epoch={epoch_time/60.0:.2f} min, ETA={eta_min:.1f} min | "
            f"train total={train_meter['total']:.4f}, coarse={train_meter['coarse_kd']:.4f}, "
            f"feat={train_meter['feat_kd']:.4f}, gt={train_meter['gt_sup']:.4f} | "
            f"val total={val_meter['total']:.4f}, coarse={val_meter['coarse_kd']:.4f}, "
            f"feat={val_meter['feat_kd']:.4f}, gt={val_meter['gt_sup']:.4f}"
        )

        with open(log_path, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                epoch,
                train_meter["total"], train_meter["coarse_kd"], train_meter["feat_kd"], train_meter["gt_sup"],
                val_meter["total"],   val_meter["coarse_kd"],   val_meter["feat_kd"],   val_meter["gt_sup"],
            ])

        save_path = run_dir / f"epoch_{epoch:02d}.pth"
        torch.save({
            "epoch": epoch, "run_name": run_name, "student_kind": student_kind,
            "model_state_dict":     model.state_dict(),
            "projector_state_dict": feature_projector.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config":     config.__dict__,
            "train_loss": train_meter,
            "val_loss":   val_meter,
        }, save_path)

    # Free this run's model before loading the next
    del model, config, feature_projector, criterion, optimizer, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nResume training finished.")


VRAM after cleanup: 77 MB allocated, 124 MB reserved
Device: cuda
Train batches: 225, Val batches: 25

Resuming run_full_hybrid from epoch 5 → 20


train:  56%|█████▌    | 126/225 [1:42:20<1:26:32, 52.45s/it, total=4.1424, feat=3.0248] 

In [12]:
# Checklist item 8: Checkpoint round-trip verification
# Reloads epoch_01.pth from Run 1 and confirms forward output is deterministic.

_ckpt_run  = "run_full_hybrid"
_ckpt_path = Path(args.output_dir) / _ckpt_run / "epoch_01.pth"
assert _ckpt_path.exists(), f"Checkpoint not found: {_ckpt_path}"

_ckpt = torch.load(_ckpt_path, map_location=device, weights_only=False)
print(f"Loaded checkpoint : {_ckpt_path}")
print(f"  epoch           : {_ckpt['epoch']}")
print(f"  student_kind    : {_ckpt['student_kind']}")
print(f"  train total loss: {_ckpt['train_loss']['total']:.4f}")

# Rebuild model + projector from saved config
_kind_r = _ckpt["student_kind"]
_model_r, _config_r, _proj_r = make_student(_kind_r, device)
for _k, _v in _ckpt["config"].items():
    if hasattr(_config_r, _k):
        setattr(_config_r, _k, _v)
_model_r.load_state_dict(_ckpt["model_state_dict"])
_proj_r.load_state_dict(_ckpt["projector_state_dict"])
_model_r.eval()
_proj_r.eval()

# Two identical forward passes — outputs must be bitwise identical
_img0_r, _img1_r, _ = next(iter(val_loader))
_img0_r = _img0_r.to(device)
_img1_r = _img1_r.to(device)

with torch.no_grad():
    _out_a = _model_r({"image0": _img0_r, "image1": _img1_r})
    _pa    = project_student_feat_for_kd(_out_a["coarse_feat0"], _proj_r)

    _out_b = _model_r({"image0": _img0_r, "image1": _img1_r})
    _pb    = project_student_feat_for_kd(_out_b["coarse_feat0"], _proj_r)

assert torch.allclose(_out_a["conf_matrix"], _out_b["conf_matrix"]), \
    "FAIL: conf_matrix differs between identical forward passes after reload"
assert torch.allclose(_pa, _pb), \
    "FAIL: projected features differ between identical forward passes after reload"

print(f"conf_matrix shape    : {_out_a['conf_matrix'].shape}")
print(f"projected feat shape : {_pa.shape}")
print("Checkpoint round-trip PASSED — reloaded model output is deterministic ✓")


Loaded checkpoint : ..\checkpoints\run_full_hybrid\epoch_01.pth
  epoch           : 1
  student_kind    : hybrid
  train total loss: 7.1253
conf_matrix shape    : torch.Size([4, 3600, 3600])
projected feat shape : torch.Size([4, 3600, 256])
Checkpoint round-trip PASSED — reloaded model output is deterministic ✓


## Team Update

Smoke test completed to verify the pipeline wiring.

- Smoke test params: `student_type=hybrid`, `num_pairs=200`, `epochs=1`, `batch_size=4`
- Full run params: `student_type=hybrid`, `num_pairs=1000`, `epochs=5`, `batch_size=4`
- Active losses: `coarse_kd`, `feat_kd`, `gt_sup` (`fine_kd` disabled)
- Verified: dataloader split, teacher hook + dual-softmax target, KD + GT loss flow, checkpoint and epoch logging.

Next: run the full training cell and compare validation loss trends from `epoch_losses.csv`.